# Bullish Harami：用 qust 识别看涨孕线

来源参考：[Investopedia](https://www.investopedia.com/terms/b/bullishharami.asp)


这篇 notebook 按 Investopedia 原页面的信息结构做完整中文改写，并把指标定义落成 `col(...).investopedia.xxx(...)` 的一行调用。能用 qust 现有 rolling、shift、select、with_cols、over 组合的就直接组合；需要 pivot/形态扫描的部分由 Rust helper 完成，Python 端不写 UDF。


## 1. Investopedia 原文内容完整改写：Bullish Harami

### 什么是 Bullish Harami
Bullish Harami 是两根蜡烛线组成的潜在看涨反转形态。第一根是较大的阴线，第二根是较小的阳线，并且第二根实体位于第一根实体内部。“Harami”可以理解为后一根小实体被前一根大实体包住。

### 出现背景
这个形态通常出现在下跌之后才有意义。第一根大阴线代表卖方仍占优势，第二根小阳线代表卖压减弱、买方开始尝试反击。它本身不是强烈反转保证，更像是下跌趋势里动能变化的早期提示。

### 结构条件
严格定义会要求前一根 close 低于 open，当前 close 高于 open，当前 open 和 close 都落在前一根实体范围内。第二根实体一般应明显小于第一根实体，否则“孕线”的含义会减弱。影线是否完全包含通常不是核心，重点仍是实体关系。

### 市场心理
第一根大阴线让市场继续相信空头占优；第二根没有继续大幅下跌，反而收出小阳线，说明卖方推动价格的能力下降。多头未必已经控制市场，但供需力量开始从单边下跌转向犹豫。

### 使用方式
交易者常等待后续 K 线确认，例如下一根继续上涨、突破第二根高点、或站上短期均线。也会结合支撑位、超卖指标或成交量。若只是孤立出现，可信度有限。

### 局限性
Bullish Harami 的信号较弱，尤其在强下跌趋势中，小阳线可能只是下跌中继。参数上，第二根实体多小才算“小”存在主观性，因此程序化时需要 `max_body_ratio` 这类阈值。

## 2. 从文章到 qust 算子的落地

qust 默认要求先有简单下跌背景，再检查前一根大阴线和后一根小阳线的实体包含关系。输出 `bullish_harami`，用户可以继续接确认条件。

## 3. qust 一行调用

```python
col("open", "high", "low", "close").investopedia.bullish_harami()
```

输入列顺序：`open, high, low, close`。

输出列：`bullish_harami`。

这些输出都保持和输入相同的行数，后面可以继续 `.with_cols(...)`、`.filter(...)`、`.monitor...`，也可以接 `.over("ticker", "ct")` 按合约独立计算。

In [1]:
import os
import sys

LOCAL_QUST_SOURCE = "/root/otters/otters-py/python"
if os.path.isdir(LOCAL_QUST_SOURCE) and LOCAL_QUST_SOURCE not in sys.path:
    sys.path.insert(0, LOCAL_QUST_SOURCE)

import qust as qs
import qust.future.future  # 注册 bt/stra/kline/fp 等金融命名空间
import qust.investopedia  # 注册 investopedia 命名空间
from qust import col, mark_shape
from qust._polars import pl

pl.Config.set_tbl_rows(16)
pl.Config.set_tbl_cols(28)

DATA_PATH = "/root/qust-py/examples/data/data_kline3.parquet"
PLOT_TICKER = "AP"


In [2]:
raw = pl.read_parquet(DATA_PATH).sort(["ticker", "ct", "datetime"])

base_contract = (
    raw
    .filter(pl.col("ticker") == PLOT_TICKER)
    .select("ct")
    .unique()
    .sort("ct")
    .get_column("ct")[0]
)

print("raw shape:", raw.shape)
print("tickers:", raw.select(pl.col("ticker").unique().sort()).to_series().to_list())
print("contract count:", raw.select("ticker", "ct").unique().height)
print("default plot ticker/ct:", PLOT_TICKER, base_contract)
raw.head(5)


raw shape: (408782, 8)
tickers: ['AP', 'RM', 'SA', 'al', 'eb', 'eg', 'fu', 'rb']
contract count: 141
default plot ticker/ct: AP 205


ticker,ct,datetime,open,high,low,close,volume
str,i32,datetime[ms],f64,f64,f64,f64,f64
"""AP""",205,2022-01-04 09:00:00,8394.0,8394.0,8392.0,8392.0,1100.0
"""AP""",205,2022-01-04 09:05:00,8385.0,8389.0,8348.0,8378.0,11169.0
"""AP""",205,2022-01-04 09:10:00,8375.0,8376.0,8298.0,8302.0,14001.0
"""AP""",205,2022-01-04 09:15:00,8301.0,8315.0,8271.0,8280.0,12839.0
"""AP""",205,2022-01-04 09:20:00,8279.0,8285.0,8243.0,8246.0,11496.0


## 4. 计算指标

下面用真实本地 K 线数据计算。对合约相关指标，示例都使用 `.over("ticker", "ct")`，表示每个品种、每个合约独立维护上下文，避免不同合约的数据串在一起。

In [3]:
indicator_expr = col("open", "high", "low", "close").investopedia.bullish_harami()
harami_data = col.with_cols(indicator_expr).over("ticker", "ct").calc_data(raw)
plot_data = (
    harami_data
    .filter((pl.col("ticker") == PLOT_TICKER) & (pl.col("ct") == base_contract))
    .sort("datetime")
    .head(1200)
)

summary = col(
    col("bullish_harami").cast(pl.UInt32).sum().alias("bullish_harami_count"),
).calc_data(harami_data)

print("plot shape:", plot_data.shape)
summary

plot shape: (1200, 9)


bullish_harami_count
u32
4464


## 5. 用 monitor 画出来

图不是静态 PNG，而是 qust monitor 输出。你可以在 notebook 里放大、拖动、查看指标与 K 线的对应关系。

In [4]:
harami_plot = col(
    col("datetime", "open", "high", "low", "close", "volume")
        .monitor("harami_price", show_axis_label=True)
        .kline(),
    col("datetime", "low", "bullish_harami")
        .monitor("harami_price", show_axis_label=True)
        .mark(shape=mark_shape.triangle_up, color="#50fa7b", width=0.45),
).monitor.make_monitor("black").monitor.add_grid([
    ["harami_price"],
]).runtime()

harami_plot.plot(plot_data, open_in_jupyter=True, auto_open=False, height=560)

## 6. Bullish Harami 策略回测

看涨孕线在当前样本里按原方向做多为正。策略在 `bullish_harami` 成立后一根 K 线做多，用 3% 止盈、1.5% 止损离场，并用 `col("hold") / col.all.fp.vol_pms()` 归一化持仓。

In [5]:
TAKE_PROFIT = 0.03
STOP_LOSS = 0.015


def make_two_sided_strategy(indicator_cols, open_long_raw, open_short_raw):
    """用当前指标生成完整多空策略；持仓用 fp.vol_pms 做品种/波动率尺度归一化。"""
    return (
        col
        .with_cols(indicator_cols)
        .with_cols(
            open_long_raw.fill_null(col.lit(False)).alias("open_long_raw"),
            open_short_raw.fill_null(col.lit(False)).alias("open_short_raw"),
        )
        # 指标在当前 K 线收盘后才确认，所以入场信号后移一根 K 线，避免同根 K 线偷看。
        .with_cols(
            col("open_long_raw").shift(1).expanding().fill_null(col.lit(False)).alias("open_long_sig"),
            col("open_short_raw").shift(1).expanding().fill_null(col.lit(False)).alias("open_short_sig"),
        )
        .with_cols(
            col("open_long_sig", "close").stra.exit_by_pct(TAKE_PROFIT, False).expanding().alias("take_profit_long"),
            col("open_long_sig", "close").stra.exit_by_pct(STOP_LOSS, True).expanding().alias("stop_loss_long"),
            col("open_short_sig", "close").stra.exit_by_pct(TAKE_PROFIT, True).expanding().alias("take_profit_short"),
            col("open_short_sig", "close").stra.exit_by_pct(STOP_LOSS, False).expanding().alias("stop_loss_short"),
        )
        .with_cols(
            (col("take_profit_long") | col("stop_loss_long") | col("open_short_sig"))
                .fill_null(col.lit(False))
                .alias("exit_long_sig"),
            (col("take_profit_short") | col("stop_loss_short") | col("open_long_sig"))
                .fill_null(col.lit(False))
                .alias("exit_short_sig"),
        )
        .with_cols(
            col("open_long_sig", "exit_long_sig", "open_short_sig", "exit_short_sig")
                .stra.to_hold_two_sides()
                .expanding()
                .alias("hold")
        )
        .with_cols(
            (col("hold") / col.all.fp.vol_pms()).alias("hold")
        )
        .with_cols(col("close", "hold").bt.price(fee_rate=0.0).expanding())
        .over("ticker", "ct")
        .select(
            col("pnl")
                .sum()
                .group_by(col("datetime").dt.date().alias("date"))
                .batch.sort("date")
                .with_cols(col("pnl").sum().expanding().alias("pnl_cum"))
                .select("date", "pnl", "pnl_cum")
        )
    )


def calc_strategy_stats(strategy_daily: pl.DataFrame) -> pl.DataFrame:
    return col(
        col("date").first_value().alias("start_date"),
        col("date").last_value().alias("end_date"),
        col.lit(1).sum().alias("days"),
        col("pnl").sum().alias("total_pnl"),
        col("pnl").mean().alias("mean_daily_pnl"),
        col("pnl").std().alias("std_daily_pnl"),
        (col("pnl").mean() / col("pnl").std() * col.lit(252 ** 0.5)).alias("sharpe_like"),
        col("pnl").min().alias("worst_day_pnl"),
        col("pnl").max().alias("best_day_pnl"),
    ).calc_data(strategy_daily)

indicator_cols = col("open", "high", "low", "close").investopedia.bullish_harami()
strategy_daily_expr = make_two_sided_strategy(
    indicator_cols,
    col("bullish_harami"),
    col.lit(False),
)
strategy_daily = strategy_daily_expr.calc_data(raw)
strategy_stats = calc_strategy_stats(strategy_daily)

print("strategy_daily shape:", strategy_daily.shape)
strategy_stats


strategy_daily shape: (859, 3)


start_date,end_date,days,total_pnl,mean_daily_pnl,std_daily_pnl,sharpe_like,worst_day_pnl,best_day_pnl
date,date,i32,f64,f64,f64,f64,f64,f64
2022-01-04,2024-12-31,859,48.920928,0.057352,1.968789,0.462431,-5.884274,5.938613


In [6]:
strategy_daily.tail(12)


date,pnl,pnl_cum
date,f64,f64
2024-12-18,-0.121247,47.593887
2024-12-19,-1.587315,46.006572
2024-12-20,0.378742,46.385314
2024-12-21,-0.125345,46.259969
2024-12-23,-0.541623,45.718346
2024-12-24,1.935891,47.654236
2024-12-25,0.089079,47.743315
2024-12-26,-0.15077,47.592545
2024-12-27,-2.118358,45.474188


## 7. 策略 PnL 曲线

下面用 qust monitor 同时画累计 PnL 和每日 PnL。累计曲线显示这套规则跨合约、跨日期后的整体资金变化；每日柱状图用来观察收益是否集中在少数日期。

In [7]:
pnl_dashboard = col(
    col("date", "pnl_cum")
        .monitor("strategy_pnl_cum", show_axis_label=True)
        .line(),
    col("date", "pnl")
        .monitor("strategy_daily_pnl", show_axis_label=True)
        .bar(),
).monitor.make_monitor("black").monitor.add_grid([
    ["strategy_pnl_cum"],
    ["strategy_daily_pnl"],
]).runtime()

pnl_dashboard.plot(strategy_daily, open_in_jupyter=True, auto_open=False, height=640)


## 8. 使用时的注意事项

- 技术指标只能把价格结构转成可计算规则，不等于确定性交易建议。
- 形态类指标通常需要后续 K 线确认；如果用于实时交易，应把确认延迟纳入回测。
- 参数越敏感，信号越多但噪声越大；参数越保守，信号更少但滞后更明显。
- 在多合约或多股票数据上使用时，优先写 `.over("ticker", "ct")` 或合适的分组键。